In [ ]:

%load_ext autoreload 
%autoreload 2
%matplotlib widget
%matplotlib inline
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
import os
from tqdm.notebook import tqdm
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.lines as mlines 
import sim_loader as sim_load
from scipy.integrate import cumtrapz
from scipy.stats import loguniform


In [ ]:
q_samples = np.geomspace(1e-6, 1e-2, 400)
fig, axs = plt.subplots(1, 1, figsize=(6, 5))
s_minus, s_plus, qs_plot = lensing_triangle_line_QS_space(q_samples, eta=0.35, xi=1/50, Amax=3000)
axs.loglog(s_plus, qs_plot, color='grey', ls='--', label='plus branch')
axs.loglog(s_minus, qs_plot, color='grey',  ls='-',  label='minus branch')
axs.set_xlabel('Projected separation s')
axs.set_ylabel('Mass ratio q')
axs.legend()


## Boxes plot with loguniform distribution in s and q
The idea is to random generate N planets from a loguniform distribution in q and s and the multiplying it by the lensing triangle to yield an occurrence rate and see if it gets close to Romna.
Then on a later step replace the loguniform distribution with my planet distribution

In [ ]:
q_min = 1e-6
q_max = 1e-2
s_min = 5e-2
s_max = 2e1
n_bins = 15
# Define the grid bins for radius and mass
s_bins = np.geomspace(s_min, s_max, n_bins)  # Log-spaced bins for projected separation  (0.1 to 10 R_E)
q_bins = np.geomspace(q_min, q_max, n_bins)    # Log-spaced bins for mass ratio (3.3 to 3330 Earth masses)

# create a loguniform distribution of planets in log q and log s
seed = 40
rng = np.random.default_rng(seed)   # pass rng or int to random_state
N = 10000
q_samples = loguniform(q_min, q_max).rvs(size=N, random_state=rng)
s_samples = loguniform(s_min, s_max).rvs(size=N, random_state=rng)
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
q_min = xi / Amax

# flag the planets that are inside the triangles and the ones that are outside
ratio = q_samples / q_min
s_plus = ratio ** eta
s_minus = ratio ** (-eta)
mask = (q_samples >= q_min) & (s_samples >= s_minus) & (s_samples <= s_plus) # True inside, False outside
sensitivity_mask = mask.astype(float)
n_true = mask.sum()            # number of True
print("Number of detectable planets:", n_true, "out of", N, "total planets.")
fig, ax = plt.subplots(figsize=(10, 8))

# Create a 2D histogram of planet counts (from the loguniform distribution)
count_ma, s_edges, q_edges = np.histogram2d(s_samples, q_samples, bins=[s_bins, q_bins])
# bin centers aligned with counts shape (ns, nq)q_samples
s_centers = 0.5*(s_edges[:-1] + s_edges[1:])
q_centers = 0.5*(q_edges[:-1] + q_edges[1:])
# N.B. passing the sensitivity mask as wieghts to histogram2d only works if 0,1 (not even sure it actually does)
# wiegths sums the weight of the samples that fall into each bin (so if a bin has 3 planets, two planets have weight 1,
# one planet has weight 0, the bin will have count 1+1+0=2). If the sensitivity mask is not 0,1, it won't work!!!!!!!
detected_counts, s_edges, q_edges = np.histogram2d(s_samples, q_samples,
                                                   bins=[s_bins, q_bins],
                                                   weights=sensitivity_mask)

# get raw counts for comparison
raw_counts, _, _ = np.histogram2d(s_samples, q_samples, bins=[s_bins, q_bins])
# Verify the detections, sensitivity mask planets and total planets
print("detected",detected_counts.sum(), "sensitivity mask", sensitivity_mask.sum(), "raw", raw_counts.sum())

# plot with pcolormesh (use edges; transpose counts to match axes)
mesh = ax.pcolormesh(s_edges, q_edges, detected_counts.T, cmap='viridis', shading='auto')
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('Detected planets (counts * sensitivity)', size = 20)

# annotate counts / percentages (use bin centers)
for i in range(s_centers.size):
    for j in range(q_centers.size):
        val = detected_counts[i, j]
        raw = raw_counts[i, j]
        #if val > 0:
        txt = f"{int(round(val))}\n({int(round(raw))})"
        ax.text(s_centers[i], q_centers[j], txt, ha='center', va='center', color='white', fontsize=8)

#overplot the lensing triangle
s_minus, s_plus, q_masked = lensing_triangle_line_QS_space(q_samples, eta, xi, Amax)
ax.loglog(s_plus, q_masked, color='grey')
ax.loglog(s_minus, q_masked, color='grey')


#  Set axis scales and labels
ax.set_ylim(1e-6, 1e-2)
ax.set_xlim(5e-2, 2e1)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('s [$R_{\mathrm{E}}$]', fontsize=25, labelpad=20)
ax.set_ylabel('q', fontsize=25, labelpad=20)
ax.set_title('Planet counts', fontsize=20)
ax.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
ax.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()
plt.savefig("figures/pop_synth/lensing_triangle_qs", bbox_inches='tight')

## Converting from (q,s) to (m,a) space
The idea is that now we random sample from a loguniform distribution in (m,a) instead of (q,s) and we plot the planet count in (m,a) space

In [ ]:
num = 10000
D_s = 8
D_l = 4
Mp_samples = np.geomspace(1e-3, 1e3, num = num)
Ml_samples = np.geomspace(1e-1*u.M_sun.to(u.M_earth), 1*u.M_sun.to(u.M_earth), num = num)
#Ml_samples = 0.5 * u.M_sun.to(u.M_earth) * np.ones_like(Mp_samples)
q = Mp_samples / Ml_samples
print("shapes:", Mp_samples.shape, Ml_samples.shape, q.shape)
fig, axs = plt.subplots(1, 1, figsize=(6, 5))
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(Mp_samples, Ml_samples, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
print("shapes:", a_plus.shape, Mp_plot.shape) 

axs.loglog(a_plus, Mp_plot, color='grey', ls='--', label='plus branch')
axs.loglog(a_minus, Mp_plot, color='grey',  ls='-',  label='minus branch')
axs.set_xlabel('Semimajor axis a [AU]')
axs.set_ylabel('Planet mass $M_p [M_{\oplus}$]')
axs.legend()
axs.set_xlim(1e-1, 1e2)
axs.set_ylim(1e-3, 1e3)


In [ ]:
N=10000000
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
q_min = xi / Amax
qmin = 1e-7
print(qmin)
print(q_min)

In [ ]:
N=10000000
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
#q_min = xi / Amax
q_min = 1e-7
D_s = 8
D_l = 4

def Ml_samples_selection(N, which_imf='Chabrier2005'):
    # random sample initial star masses from the IMF of Penny (disc)
    if which_imf in ['Chabrier2005', 'Kroupa', 'Robin_2003']:
        Mstars = np.geomspace(1e-1, 1e2, N)
        IMF_pdf = np.zeros(N)
        MC_random = np.random.uniform(0, 1, N)

        for i in range(0, N):
            if which_imf == 'Chabrier2005':
                IMF_pdf[i] = Chabrier_2005_IMF_pdf(Mstars[i])
            if which_imf == 'Kroupa':
                IMF_pdf[i] = Kroupa_IMF_pdf(Mstars[i])    
            if which_imf == 'Robin_2003':
                IMF_pdf[i] = Robin_2003_IMF(Mstars[i])


        # Assume x is your array (can be linear or log-spaced), pdf is the unnormalized PDF
        dx = np.diff(Mstars)
        dx = np.append(dx, dx[-1])  # Make dx same length as x
        # Compute normalization constant (area under curve)
        area = np.sum(IMF_pdf * dx)
        # Normalize
        IMF_pdf_norm = IMF_pdf / area
        # sample the cdf from the normalized PDF
        IMF_cdf = cumtrapz(IMF_pdf_norm, Mstars, initial=0)
        IMF_cdf /= IMF_cdf[-1]
        Ml_samples = np.interp(MC_random, IMF_cdf, Mstars) * const.M_sun.to(u.M_earth).value

    elif which_imf == 'loguniform':
        M_l_min = 0.1 * u.M_sun.to(u.M_earth)
        M_l_max = 100 * u.M_sun.to(u.M_earth)
        seed = 3
        rng = np.random.default_rng(seed)   # pass rng or int to random_state
        Ml_samples = loguniform(M_l_min, M_l_max).rvs(size=N, random_state=rng)
    elif which_imf == 'constant':
        Ml_samples = 0.2 * u.M_sun.to(u.M_earth) * np.ones(N)
    return Ml_samples

Ml_samples = Ml_samples_selection(N, which_imf='Robin_2003')

# create a loguniform distribution of planets in log m and log a
a_min = 1e-2
a_max = 1e2
m_min = 1e-2
m_max = 1e3
seed = 2
rng = np.random.default_rng(seed)   # pass rng or int to random_state
m_samples = loguniform(m_min, m_max).rvs(size=N, random_state=rng)
a_samples = loguniform(a_min, a_max).rvs(size=N, random_state=rng)
n_bins = 15
# Define the grid bins for radius and mass
a_bins = np.geomspace(a_min, a_max, n_bins)  # Log-spaced bins for semimajor-axis (1e-1, 1e2) AU
m_bins = np.geomspace(m_min, m_max, n_bins)  # Log-spaced bins for mass  (1e-3, 1e3 Earth masses)

# convert (M,a) to (q,s) per sample
q_samples = m_samples / Ml_samples
s_samples = a_to_s(a_samples, Ml_samples, D_s, D_l)


# flag the planets that are inside the triangles and the ones that are outside
ratio = q_samples / q_min
s_plus = ratio ** eta
s_minus = ratio ** (-eta)
# compute a_minus/a_plus using the matching lens masses
a_minus = s_minus * R_Einstein(Ml_samples, D_s, D_l)
a_plus = s_plus * R_Einstein(Ml_samples, D_s, D_l)
mask = (m_samples >= q_min * Ml_samples) & (a_samples >= a_minus) & (a_samples <= a_plus) # True inside, False outside
sensitivity_mask = mask.astype(float)
n_true = mask.sum()            # number of True
print("Number of detectable planets:", n_true, "out of", N, "total planets.")
fig, ax = plt.subplots(figsize=(13, 9))

# Create a 2D histogram of planet counts (from the loguniform distribution)
count_ma, a_edges, m_edges = np.histogram2d(a_samples, m_samples, bins=[a_bins, m_bins])
# bin centers aligned with counts shape (ns, nq)
a_centers = 0.5*(a_edges[:-1] + a_edges[1:])
m_centers = 0.5*(m_edges[:-1] + m_edges[1:])
# N.B. passing the sensitivity mask as wieghts to histogram2d only works if 0,1 (not even sure it actually does)
# wiegths sums the weight of the samples that fall into each bin (so if a bin has 3 planets, two planets have weight 1,
# one planet has weight 0, the bin will have count 1+1+0=2). If the sensitivity mask is not 0,1, it won't work!!!!!!!
detected_counts, a_edges, m_edges = np.histogram2d(a_samples, m_samples,
                                                   bins=[a_bins, m_bins],
                                                   weights=sensitivity_mask)

# get raw counts for comparison
raw_counts, _, _ = np.histogram2d(a_samples, m_samples, bins=[a_bins, m_bins])
# Verify the detections, sensitivity mask planets and total planets
print("detected",detected_counts.sum(), "sensitivity mask", sensitivity_mask.sum(), "raw", raw_counts.sum())

# plot with pcolormesh (use edges; transpose counts to match axes)
mesh = ax.pcolormesh(a_edges, m_edges, detected_counts.T, cmap='viridis', shading='auto')
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('Detected planets (counts * sensitivity)', size = 20)

# annotate counts / percentages (use bin centers)
for i in range(a_centers.size):
    for j in range(m_centers.size):
        val = detected_counts[i, j]
        raw = raw_counts[i, j]
        #if val > 0:
        txt = f"{int(round(val))}\n({int(round(raw))})"
        ax.text(a_centers[i], m_centers[j], txt, ha='center', va='center', color='white', fontsize=8)

# # trying to plot the lensing triangle
ML = 0.1 * u.M_sun.to(u.M_earth) * np.ones_like(m_samples)
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(m_samples, ML, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
ax.loglog(a_plus, Mp_plot, color='grey')
ax.loglog(a_minus, Mp_plot, color='grey')
ML = 1 * u.M_sun.to(u.M_earth) * np.ones_like(m_samples)
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(m_samples, ML, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
ax.loglog(a_plus, Mp_plot, color='grey', ls = '--')
ax.loglog(a_minus, Mp_plot, color='grey', ls = '--')
plot.plot_roman_sensitivity(fig, ax, roman_sensitivity=True, solar_system= False, ss_moons= False, kepler=False)
#  Set axis scales and labels
ax.set_xlim(a_min, a_max)
ax.set_ylim(m_min, m_max)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('a [AU]', fontsize=25, labelpad=20)
ax.set_ylabel('$M_{\mathrm{P}} [M_{\oplus}]$', fontsize=25, labelpad=20)
ax.set_title('Planet counts', fontsize=20)
ax.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
ax.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()
plt.savefig("figures/pop_synth/lensing_triangle_MA_space_Mstar_robin_2003_overplot_roman_qe-7", bbox_inches='tight')

## Same plot as above in q,s space

## Roman plot with Penny sensitivity

In [ ]:
N = 100
m_min = 1e-3
m_max = 1e3
M_p = loguniform(m_min, m_max).rvs(size=N, random_state=rng)
M_l_min = 0.1 * u.M_sun.to(u.M_earth)
M_l_max = 1.0 * u.M_sun.to(u.M_earth)
seed = 7
rng = np.random.default_rng(seed)   # pass rng or int to random_state
M_l = loguniform(M_l_min, M_l_max).rvs(size=N, random_state=rng)
q = M_p / M_l
order = np.argsort(q)
q = q[order]
M_l = M_l[order]
M_p = M_p[order]

q_min = xi / Amax
ratio = q / q_min
s_plus = ratio ** eta
s_minus = ratio ** (-eta)

mask = q >= q_min 
q_masked = q[mask] # select only the masses that have q>=q_min
s_plus_masked = s_plus[mask] #compute the separation for those masses
s_minus_masked = s_minus[mask]
M_l_masked = M_l[mask] #corresponding lens masses
M_p_masked = M_p[mask]

# compute a_minus/a_plus using the matching (masked) lens masses
a_minus = s_minus_masked * R_Einstein(M_l_masked, D_s, D_l)
a_plus = s_plus_masked * R_Einstein(M_l_masked, D_s, D_l)

# M_p_masked computed from q_masked * corresponding M_l_masked
#M_p_masked = q_masked * M_l_masked

fig, axs = plt.subplots(1, 1, figsize=(6, 5))
# axs.loglog(M_p)
# axs.loglog(M_l)
# axs.loglog(q)
# axs.loglog(s_minus_masked)
axs.loglog(a_plus, M_p_masked, color='grey',  label='random star mass')
axs.loglog(a_minus, M_p_masked, color='grey')

M_l = 0.1 * u.M_sun.to(u.M_earth) * np.ones_like(M_p)
a_plus, a_minus, M_p_masked = lensing_triangle_line_MA_space(M_p, M_l, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
axs.loglog(a_plus, M_p_masked, color='gold')
axs.loglog(a_minus, M_p_masked, color='gold',  ls='-',  label='constant $M_{\star} = 0.1 \: M_{\odot}$ ')

M_l = 1 * u.M_sun.to(u.M_earth) * np.ones_like(M_p)
a_plus, a_minus, M_p_masked = lensing_triangle_line_MA_space(M_p, M_l, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
axs.loglog(a_plus, M_p_masked, color='green')
axs.loglog(a_minus, M_p_masked, color='green',  ls='-',  label='constant $M_{\star} = 1 \: M_{\odot}$ ')

axs.set_ylim(1e-2,1e3)
axs.set_xlabel('Semimajor axis a [AU]', size = 15)
axs.set_ylabel('Planet mass $M_p [M_{\oplus}$]', size = 15)
axs.set_title('Lensing triangle with random sampling of $M_{\mathrm{p}}$ and $M_{\mathrm{L}}$')
axs.tick_params(axis="both", which="major", direction='out', size=13, labelsize=12)
axs.tick_params(axis="both", which="minor", direction='out', size=8)

axs.legend()
plt.savefig("figures/pop_synth/lensing_triangle_line_MA_space_debug", bbox_inches='tight')

In [ ]:
fig, axs = plt.subplots(figsize=(10, 8))
smap = np.loadtxt('all.magrid.NRO.layout_7f_3_covfac.52.filled') 
x = 10**smap[:33,0]
y = 10**smap[::33,1]
print(smap.shape,x.shape[0]*y.shape[0])
X,Y=np.meshgrid(x,y)
z = 10**smap[:,2].reshape(X.shape) 
a_min = 1e-1
a_max = 1e2
m_min = 1e-3
m_max = 1e3
m_samples = loguniform(m_min, m_max).rvs(size=N, random_state=rng)
a_samples = loguniform(a_min, a_max).rvs(size=N, random_state=rng)
# Extend the bin edges for x and y
x_bins = np.append(x, x[-1] * 1.1)  # Add an extra bin edge for x
y_bins = np.append(y[::-1], y[::-1][-1] * 1.1)  # Add an extra bin edge for y
counts, _, _ = np.histogram2d(a_samples, m_samples, bins=[x_bins, y_bins])
counts = np.fliplr(counts)# flips the count left
print(counts.shape)
pc = axs.pcolormesh(X, Y, counts.T*z, cmap='viridis', shading='auto')  # Transpose counts to match X, Y
cbar = plt.colorbar(pc, ax=axs, label='Number of Planets detection if 1 planet per star')# Ensure z matches the histogram grid
axs.set_xlabel('a [AU]', fontsize=25, labelpad=20)
axs.set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)
axs.set_xscale('log')
axs.set_yscale('log')

axs.set_xlim(1e-1, 1e2)
axs.set_ylim(1e-3, 1e3)
plt.savefig("figures/pop_synth/microlensing_sensitivity_map_roman", bbox_inches='tight')


## Weighted planet count with my simulations

In [ ]:
# Folder paths for t5 and t3
folder_path = [
    "sims/gas_acc/stellar_masses/single_planets/linear/vfrag/surfheat/Fe_H_07/5Myrs_randomMstar_randomZ_randomTau",
]


timestep = 500
H_r_model = ['Lambrechts_mixed']*1000
Mstar = [1.]  # Stellar mass in M_sun
simulations = []
sim_parameters = []
parameters = []
planet_dicts = []
# Function to load simulations and compute planet_dict
def load_simulations_and_planet_dict(folder_paths, H_r_model, Mstar, timestep, simulations, parameters, sim_parameters):
    for folder_path, H_r_model, mstar in zip(folder_paths, H_r_model, Mstar):
        # List all files in the given folder
        all_files = os.listdir(folder_path)
        # Create the list of names of the sim, sim_params, and params files
        sim_filenames = [os.path.join(folder_path, f) for f in all_files if f.startswith('simulation')]
        sim_params_filenames = [os.path.join(folder_path, f) for f in all_files if f.startswith('sim_params') ]
        params_filenames = [os.path.join(folder_path, f) for f in all_files if f.startswith('params')]
        print("Number of files in folder:", len(all_files))
        print("Number of simulation files:", len(sim_filenames))
        # Sort the filenames based on the initial time
        sim_filenames.sort(key=sim_load.extract_initial_time)
        sim_params_filenames.sort(key=sim_load.extract_initial_time)
        params_filenames.sort(key=sim_load.extract_initial_time)

        # Load the simulations, sim_params, and params
        simulations.append([sim_load.JSONtoSimRes(filename) for filename in sim_filenames])
        sim_parameters.append([sim_load.load_sim_params(filename) for filename in sim_params_filenames])
        params = [sim_load.load_params(filename) for filename in params_filenames]
        parameters.append(params)
        print("Number of simulations loaded:", len(simulations[-1]))

        # Compute planet_dict for the folder
        planet_dicts.append(planet_counter(simulations[-1], parameters[-1], sim_parameters[-1], outer=False))
        print('planet dict:', planet_dicts[-1])



simulations = []
sim_parameters = []
parameters = []
load_simulations_and_planet_dict(folder_path, H_r_model, Mstar, timestep, simulations, parameters, sim_parameters)
print(len(simulations[0]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# final positions and masses of the simulation loaded above
final_positions = []
final_masses = []
star_masses = []
for j in range(len(simulations[0])):
    sim = simulations[0][j]
    params = parameters[0][j]
    sim_params = sim_parameters[0][j]
    for p in range(sim_params.nr_planets):
        idx = plot.idxs (axs, sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p], 
                        sim.dR_dt[p], sim.dM_dt[p], params, True)
        iso_idx = np.argmax(sim.mass[p] > M_peb_iso(H_R(sim.position[p].value, M_dot_star(sim.time.value, params), params), params)*u.M_earth)
        stop_mig_idx = idx['stop_mig_idx'].values[0]

        final_positions.append(sim.position[p,stop_mig_idx].to(u.au).value)
        final_masses.append(sim.mass[p,stop_mig_idx].to(u.M_earth).value) 
        star_masses.append(params.star_mass)

final_positions = np.array(final_positions)
final_masses = np.array(final_masses)
star_masses = np.array(star_masses)
N=10000
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
q_min = xi / Amax
D_s = 8
D_l = 4

# create a loguniform distribution of planets in log m and log a
a_min = 1e-2
a_max = 1e2
m_min = 1e-2
m_max = 1e3
n_bins = 10
# Define the grid bins for radius and mass
a_bins = np.geomspace(a_min, a_max, n_bins)  # Log-spaced bins for semimajor-axis (1e-2, 1e2) AU
m_bins = np.geomspace(m_min, m_max, n_bins)  # Log-spaced bins for mass  (1e-2, 1e3 Earth masses)


# convert (M,a) to (q,s) per sample
q_samples = final_masses / star_masses
s_samples = a_to_s(final_positions, star_masses, D_s, D_l)


# flag the planets that are inside the triangles and the ones that are outside
ratio = q_samples / q_min
s_plus = ratio ** eta
s_minus = ratio ** (-eta)
# compute a_minus/a_plus using the matching lens masses
a_minus = s_minus * R_Einstein(star_masses, D_s, D_l)
a_plus = s_plus * R_Einstein(star_masses, D_s, D_l)
mask = (final_masses >= q_min * star_masses) & (final_positions >= a_minus) & (final_positions <= a_plus) # True inside, False outside
sensitivity_mask = mask.astype(float)
n_true = mask.sum()            # number of True
print("Number of detectable planets:", n_true, "out of", N, "total planets.")
fig, ax = plt.subplots(figsize=(10, 8))

# Create a 2D histogram of planet counts (from the loguniform distribution)
count_ma, a_edges, m_edges = np.histogram2d(final_positions, final_masses, bins=[a_bins, m_bins])
# bin centers aligned with counts shape (ns, nq)
a_centers = 0.5*(a_edges[:-1] + a_edges[1:])
m_centers = 0.5*(m_edges[:-1] + m_edges[1:])
# N.B. passing the sensitivity mask as wieghts to histogram2d only works if 0,1 (not even sure it actually does)
# wiegths sums the weight of the samples that fall into each bin (so if a bin has 3 planets, two planets have weight 1,
# one planet has weight 0, the bin will have count 1+1+0=2). If the sensitivity mask is not 0,1, it won't work!!!!!!!
detected_counts, a_edges, m_edges = np.histogram2d(final_positions, final_masses,
                                                   bins=[a_bins, m_bins],
                                                   weights=sensitivity_mask)

# get raw counts for comparison
raw_counts, _, _ = np.histogram2d(final_positions, final_masses, bins=[a_bins, m_bins])
# Verify the detections, sensitivity mask planets and total planets
print("detected",detected_counts.sum(), "sensitivity mask", sensitivity_mask.sum(), "raw", raw_counts.sum())

# plot with pcolormesh (use edges; transpose counts to match axes)
mesh = ax.pcolormesh(a_edges, m_edges, detected_counts.T, cmap='viridis', shading='auto', norm=LogNorm(vmin=np.min(detected_counts[detected_counts>0]), vmax=np.max(detected_counts)))
cbar = fig.colorbar(mesh, ax=ax, norm = norm)
cbar.set_label('Detected planets (counts * sensitivity)', size = 20)
cbar.ax.tick_params(labelsize=20) 
cbar.ax.tick_params(axis="both", which="major", size=13, labelsize=13)
cbar.ax.tick_params(axis="both", which="minor", size=9)
# annotate counts / percentages (use bin centers)
for i in range(a_centers.size):
    for j in range(m_centers.size):
        val = detected_counts[i, j]
        raw = raw_counts[i, j]
        #if val > 0:
        txt = f"{int(round(val))}\n({int(round(raw))})"
        ax.text(a_centers[i], m_centers[j], txt, ha='center', va='center', color='white', fontsize=8)

# # trying to plot the lensing triangle
ML = 0.01 * u.M_sun.to(u.M_earth) * np.ones_like(final_masses)
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(final_masses, ML, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
ax.loglog(a_plus, Mp_plot, color='grey')
ax.loglog(a_minus, Mp_plot, color='grey')
ML = 100 * u.M_sun.to(u.M_earth) * np.ones_like(final_masses)
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(final_masses, ML, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
ax.loglog(a_plus, Mp_plot, color='grey', ls = '--')
ax.loglog(a_minus, Mp_plot, color='grey', ls = '--')

#  Set axis scales and labels
ax.set_xlim(1e-2, 1e2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('a [AU]', fontsize=25, labelpad=20)
ax.set_ylabel('$M_{\mathrm{P}} [M_{\oplus}]$', fontsize=25, labelpad=20)
ax.set_title('Planet counts', fontsize=20)
ax.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
ax.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()

r = np.geomspace(1e-2, 1e2, num =100)
ax.loglog(r, M_peb_iso(H_R(r, M_dot_star(sim.time[0].value, params), params), params), linestyle='--', color='purple')
ax.loglog(r, M_peb_iso(H_R(r, M_dot_star(sim.time[-1].value, params), params), params), linestyle='--', color='purple')
plt.savefig("figures/pop_synth/planet_count_randomZ_randomMstar_randomTau_theoretical_triangle", bbox_inches='tight')


########### PLOT WITHOUT SENSITIVITY MASK ###########
# Create a 2D histogram of planet counts
counts, xedges, yedges = np.histogram2d(final_positions, final_masses, bins=[a_bins, m_bins])

# Normalize counts to percentages
print(planet_dicts)
total_planets = planet_dicts[0]['tot_planets']
print("Total planets:", total_planets)
percentages = (counts / total_planets) * 100

# Plot the 2D histogram as a color-coded grid
fig, ax = plt.subplots(figsize=(10, 8))
mesh = ax.pcolormesh(xedges, yedges, counts.T, cmap='viridis', norm=LogNorm(vmin=np.min(counts[counts>0]), vmax=np.max(counts)))  # Transpose counts to match X, Y
cbar = plt.colorbar(mesh, ax=ax)
cbar.set_label('Number of Planets', fontsize=20)  # Increase colorbar label size
cbar.ax.tick_params(labelsize=20) 
cbar.ax.tick_params(axis="both", which="major", size=13, labelsize=13)
cbar.ax.tick_params(axis="both", which="minor", size=9)
# Add percentage text to each bin
for i in range(len(xedges) - 1):
    for j in range(len(yedges) - 1):
        if counts[i, j] > 0:  # Only display text for non-empty bins
            ax.text(
                (xedges[i] + xedges[i + 1]) / 2,  # x-coordinate (bin center)
                (yedges[j] + yedges[j + 1]) / 2,  # y-coordinate (bin center)
                f"{percentages[i, j]:.1f}%",      # Percentage text
                color="white",
                ha="center",
                va="center",
                fontsize=8
            )

#plot.boxes(ax)
# Set axis scales and labels
r = np.geomspace(1e-2, 1e2, num =100)
ax.loglog(r, M_peb_iso(H_R(r, M_dot_star(sim.time[0].value, params), params), params), linestyle='--', color='purple')
ax.loglog(r, M_peb_iso(H_R(r, M_dot_star(sim.time[-1].value, params), params), params), linestyle='--', color='purple')
ax.set_ylim(1e-2, 1e3)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('a [AU]', fontsize=25, labelpad=20)
ax.set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)
ax.set_title('Planet counts', fontsize=20)
ax.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
ax.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()
plt.savefig("figures/pop_synth/planet_radius_mass_distribution_surfheat_randomMstar_randomZ_linear", bbox_inches='tight')

## Random sample of Mstar, Z, disc spread


In [ ]:
# Folder paths for t5 and t3
folder_path = [
    "sims/gas_acc/stellar_masses/single_planets/linear/vfrag/surfheat/Fe_H_07/5Myrs_randomMstar_randomZ_randomTau",
]


timestep = 5000
H_r_model = ['Lambrechts_mixed']*2000
simulations = []
sim_parameters = []
parameters = []
planet_dicts = []
# Function to load simulations and compute planet_dict
def load_simulations_and_planet_dict(folder_paths, H_r_model, Mstar, timestep, simulations, parameters, sim_parameters):
    for folder_path, H_r_model, mstar in zip(folder_paths, H_r_model, Mstar):
        # List all files in the given folder
        all_files = os.listdir(folder_path)
        # Create the list of names of the sim, sim_params, and params files
        sim_filenames = [os.path.join(folder_path, f) for f in all_files if f.startswith('simulation_'+H_r_model)]
        sim_params_filenames = [os.path.join(folder_path, f) for f in all_files if f.startswith('sim_params_'+H_r_model) ]
        params_filenames = [os.path.join(folder_path, f) for f in all_files if f.startswith('params_'+H_r_model)]
        print("Number of files in folder:", len(all_files))
        print("Number of simulation files:", len(sim_filenames))
        # Sort the filenames based on the initial time
        sim_filenames.sort(key=sim_load.extract_initial_time)
        sim_params_filenames.sort(key=sim_load.extract_initial_time)
        params_filenames.sort(key=sim_load.extract_initial_time)

        # Load the simulations, sim_params, and params
        simulations.append([sim_load.JSONtoSimRes(filename) for filename in sim_filenames])
        sim_parameters.append([sim_load.load_sim_params(filename) for filename in sim_params_filenames])
        params = [sim_load.load_params(filename) for filename in params_filenames]
        parameters.append(params)
        print("Number of simulations loaded:", len(simulations[-1]))

        # Compute planet_dict for the folder
        planet_dicts.append(planet_counter(simulations[-1], parameters[-1], sim_parameters[-1], outer=False))
        print('planet dict:', planet_dicts[-1])



simulations = []
sim_parameters = []
parameters = []
load_simulations_and_planet_dict(folder_path, H_r_model, Mstar, timestep, simulations, parameters, sim_parameters)
print(len(simulations[0]))

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(8,8))

for j in range(len(simulations[0])):
    sim = simulations[0][j]
    params = parameters[0][j]
    sim_params = sim_parameters[0][j]
    for p in range(sim_params.nr_planets):
        idx = plot.idxs (axs, sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p], 
                        sim.dR_dt[p], sim.dM_dt[p], params, True)
        iso_idx = np.argmax(sim.mass[p] > M_peb_iso(H_R(sim.position[p].value, M_dot_star( sim.time.value, params), params), params)*u.M_earth)
        stop_mig_idx = idx['stop_mig_idx'].values[0]
        cmap =  mpl.cm.inferno.reversed()
        norm = mpl.colors.LogNorm(vmin = sim_params.t_in, vmax = sim_params.t_fin)
        colors = cmap(norm(sim_params.t0[p]))

        sc = axs.scatter(sim.position[p,0].to(u.au), sim.mass[p,0].to(u.M_earth), facecolors='none',edgecolors = colors, norm=norm, 
        cmap = cmap)

        # if iso_idx != 0:
        #     axs.scatter(sim.position[p,iso_idx].to(u.au), sim.mass[p,iso_idx].to(u.M_earth),
        #                     color =  cmap(norm(sim.time[iso_idx].to(u.Myr).value)), facecolors='none', marker = 'v')
        axs.scatter(sim.position[p,stop_mig_idx].to(u.au), sim.mass[p,stop_mig_idx].to(u.M_earth), 
                    color =cmap(norm(sim.time[stop_mig_idx].to(u.Myr).value)))
# Define the colormap and normalization
cmap = mpl.cm.inferno.reversed()
norm = mpl.colors.LogNorm(vmin=sim_params.t_in, vmax=sim_params.t_fin)

# Create a standalone colorbar
cbar_ax = fig.add_axes([0.95, 0.15, 0.03, 0.7])  # Position of the colorbar
cbar = mpl.colorbar.ColorbarBase(cbar_ax, cmap=cmap, norm=norm)

# Customize the colorbar
cbar_ax.yaxis.set_major_locator(mpl.ticker.LogLocator(base=10.0, subs=[1.0, 5.0]))
cbar_ax.yaxis.set_major_formatter(mpl.ticker.LogFormatter())
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(plot.custom_log_formatter))
cbar.set_label('Accretion timescale [Myr]', fontsize=20, labelpad=15)
cbar.ax.tick_params(axis="both", which="major", size=13, labelsize=13)
cbar.ax.tick_params(axis="both", which="minor", size=9)

plot.boxes(axs)
plot.plot_roman_sensitivity(fig, axs, solar_system= False, ss_moons= False, kepler=False)
#plot the initial mass line
from matplotlib import pyplot as plt, ticker as mticker
alpha_transp=0.2

axs.set_xlabel('r [AU]', fontsize = 25, labelpad=20)
axs.set_xlim(5e-3, 1e2)
axs.tick_params(axis = "both", which = "major", direction = 'in', size = 15, labelsize = 18)
axs.tick_params(axis = "both", which = "minor", direction = 'in', size = 10)
axs.set_ylim(1e-8, 1e3)
axs.set_xscale('log')
axs.set_yscale('log')
axs.set_title("Surfheat, random $M_{\star}$, random Z, random "+r"$\tau_{\mathrm{disc}}$", fontsize = 18)

plot.all_x_ticks(axs, num_ticks=100)

axs.set_ylabel('M [$M_{\oplus}$]', fontsize = 25, labelpad=20)
plt.savefig("figures/pop_synth/pop_synt_starmass_Z_tau_random", bbox_inches='tight')

## Attempt to revert Roman sensitivity
trying to get the sensitivity of Roman assuming they used a loguniform distribution in planets (divide the sensitivity by the nr of planets).

In [ ]:
N=10000000
#lensing triangle values
eta = 0.35
xi = 1/50
Amax = 3000
#q_min = xi / Amax
q_min = 1e-7
D_s = 8
D_l = 4

def Ml_samples_selection(N, which_imf='Chabrier2005'):
    # random sample initial star masses from the IMF of Penny (disc)
    if which_imf in ['Chabrier2005', 'Kroupa', 'Robin_2003']:
        Mstars = np.geomspace(1e-1, 1e2, N)
        IMF_pdf = np.zeros(N)
        MC_random = np.random.uniform(0, 1, N)

        for i in range(0, N):
            if which_imf == 'Chabrier2005':
                IMF_pdf[i] = Chabrier_2005_IMF_pdf(Mstars[i])
            if which_imf == 'Kroupa':
                IMF_pdf[i] = Kroupa_IMF_pdf(Mstars[i])    
            if which_imf == 'Robin_2003':
                IMF_pdf[i] = Robin_2003_IMF(Mstars[i])


        # Assume x is your array (can be linear or log-spaced), pdf is the unnormalized PDF
        dx = np.diff(Mstars)
        dx = np.append(dx, dx[-1])  # Make dx same length as x
        # Compute normalization constant (area under curve)
        area = np.sum(IMF_pdf * dx)
        # Normalize
        IMF_pdf_norm = IMF_pdf / area
        # sample the cdf from the normalized PDF
        IMF_cdf = cumtrapz(IMF_pdf_norm, Mstars, initial=0)
        IMF_cdf /= IMF_cdf[-1]
        Ml_samples = np.interp(MC_random, IMF_cdf, Mstars) * const.M_sun.to(u.M_earth).value

    elif which_imf == 'loguniform':
        M_l_min = 0.1 * u.M_sun.to(u.M_earth)
        M_l_max = 100 * u.M_sun.to(u.M_earth)
        seed = 3
        rng = np.random.default_rng(seed)   # pass rng or int to random_state
        Ml_samples = loguniform(M_l_min, M_l_max).rvs(size=N, random_state=rng)
    elif which_imf == 'constant':
        Ml_samples = 0.2 * u.M_sun.to(u.M_earth) * np.ones(N)
    return Ml_samples

Ml_samples = Ml_samples_selection(N, which_imf='Robin_2003')

# create a loguniform distribution of planets in log m and log a
a_min = 1e-2
a_max = 1e2
m_min = 1e-2
m_max = 1e3
seed = 2
rng = np.random.default_rng(seed)   # pass rng or int to random_state
m_samples = loguniform(m_min, m_max).rvs(size=N, random_state=rng)
a_samples = loguniform(a_min, a_max).rvs(size=N, random_state=rng)
n_bins = 15
# Define the grid bins for radius and mass
a_bins = np.geomspace(a_min, a_max, n_bins)  # Log-spaced bins for semimajor-axis (1e-1, 1e2) AU
m_bins = np.geomspace(m_min, m_max, n_bins)  # Log-spaced bins for mass  (1e-3, 1e3 Earth masses)

# convert (M,a) to (q,s) per sample
q_samples = m_samples / Ml_samples
s_samples = a_to_s(a_samples, Ml_samples, D_s, D_l)


# flag the planets that are inside the triangles and the ones that are outside
ratio = q_samples / q_min
s_plus = ratio ** eta
s_minus = ratio ** (-eta)
# compute a_minus/a_plus using the matching lens masses
a_minus = s_minus * R_Einstein(Ml_samples, D_s, D_l)
a_plus = s_plus * R_Einstein(Ml_samples, D_s, D_l)
mask = (m_samples >= q_min * Ml_samples) & (a_samples >= a_minus) & (a_samples <= a_plus) # True inside, False outside
sensitivity_mask = mask.astype(float)
n_true = mask.sum()            # number of True
print("Number of detectable planets:", n_true, "out of", N, "total planets.")
fig, ax = plt.subplots(figsize=(13, 9))

# Create a 2D histogram of planet counts (from the loguniform distribution)
count_ma, a_edges, m_edges = np.histogram2d(a_samples, m_samples, bins=[a_bins, m_bins])
# bin centers aligned with counts shape (ns, nq)
a_centers = 0.5*(a_edges[:-1] + a_edges[1:])
m_centers = 0.5*(m_edges[:-1] + m_edges[1:])
# N.B. passing the sensitivity mask as wieghts to histogram2d only works if 0,1 (not even sure it actually does)
# wiegths sums the weight of the samples that fall into each bin (so if a bin has 3 planets, two planets have weight 1,
# one planet has weight 0, the bin will have count 1+1+0=2). If the sensitivity mask is not 0,1, it won't work!!!!!!!
detected_counts, a_edges, m_edges = np.histogram2d(a_samples, m_samples,
                                                   bins=[a_bins, m_bins],
                                                   weights=sensitivity_mask)

# get raw counts for comparison
raw_counts, _, _ = np.histogram2d(a_samples, m_samples, bins=[a_bins, m_bins])
# Verify the detections, sensitivity mask planets and total planets
print("detected",detected_counts.sum(), "sensitivity mask", sensitivity_mask.sum(), "raw", raw_counts.sum())

# plot with pcolormesh (use edges; transpose counts to match axes)
mesh = ax.pcolormesh(a_edges, m_edges, detected_counts.T, cmap='viridis', shading='auto')
cbar = fig.colorbar(mesh, ax=ax)
cbar.set_label('Detected planets (counts * sensitivity)', size = 20)

# annotate counts / percentages (use bin centers)
for i in range(a_centers.size):
    for j in range(m_centers.size):
        val = detected_counts[i, j]
        raw = raw_counts[i, j]
        #if val > 0:
        txt = f"{int(round(val))}\n({int(round(raw))})"
        ax.text(a_centers[i], m_centers[j], txt, ha='center', va='center', color='white', fontsize=8)

# # trying to plot the lensing triangle
ML = 0.1 * u.M_sun.to(u.M_earth) * np.ones_like(m_samples)
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(m_samples, ML, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
ax.loglog(a_plus, Mp_plot, color='grey')
ax.loglog(a_minus, Mp_plot, color='grey')
ML = 1 * u.M_sun.to(u.M_earth) * np.ones_like(m_samples)
a_minus, a_plus, Mp_plot = lensing_triangle_line_MA_space(m_samples, ML, D_s, D_l, eta=0.35, xi=1/50, Amax=3000)
ax.loglog(a_plus, Mp_plot, color='grey', ls = '--')
ax.loglog(a_minus, Mp_plot, color='grey', ls = '--')
plot.plot_roman_sensitivity(fig, ax, roman_sensitivity=True, solar_system= False, ss_moons= False, kepler=False)
#  Set axis scales and labels
ax.set_xlim(a_min, a_max)
ax.set_ylim(m_min, m_max)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('a [AU]', fontsize=25, labelpad=20)
ax.set_ylabel('$M_{\mathrm{P}} [M_{\oplus}]$', fontsize=25, labelpad=20)
ax.set_title('Planet counts', fontsize=20)
ax.tick_params(axis="both", which="major", direction='out', size=15, labelsize=18)
ax.tick_params(axis="both", which="minor", direction='out', size=10)
plt.tight_layout()
plt.savefig("figures/pop_synth/lensing_triangle_MA_space_Mstar_robin_2003_overplot_roman_qe-7", bbox_inches='tight')